# 09 — Buyer Reliability & Risk Classification
Task 10: transaction history → completion/cancellation/dispute/payment features → transparent weighted reliability score → risk classification → feeds the final market recommendation.

## Cell 1 — Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

## Cell 2 — Create/load transaction data
Clearly-labeled synthetic prototype data — replace with real buyer transaction records as soon as they exist.

In [ ]:
np.random.seed(42)

n_buyers = 100

buyer_data = pd.DataFrame({
    "buyer_id": [
        f"B{i:03d}" for i in range(1, n_buyers + 1)
    ],
    "market": np.random.choice(
        ["Market_A", "Market_B", "Market_C", "Market_D"],
        n_buyers
    ),
    "total_orders": np.random.randint(
        20, 300, n_buyers
    ),
    "completed_orders": np.random.randint(
        15, 280, n_buyers
    ),
    "cancelled_orders": np.random.randint(
        0, 30, n_buyers
    ),
    "payment_delays": np.random.randint(
        0, 25, n_buyers
    ),
    "disputes": np.random.randint(
        0, 10, n_buyers
    ),
    "average_payment_delay_days": np.round(
        np.random.uniform(0, 10, n_buyers),
        2
    )
})

buyer_data.head()

In [ ]:
print(
    "WARNING: Transaction data is synthetic prototype data."
)

## Cell 3 — Validate the data

In [ ]:
print(buyer_data.info())
print(buyer_data.isnull().sum())

In [ ]:
assert (
    buyer_data["completed_orders"]
    <= buyer_data["total_orders"]
).all()

## Cell 4 — Completion rate

In [ ]:
buyer_data["completion_rate"] = (
    buyer_data["completed_orders"]
    / buyer_data["total_orders"]
)

## Cell 5 — Cancellation rate
Lower is better.

In [ ]:
buyer_data["cancellation_rate"] = (
    buyer_data["cancelled_orders"]
    / buyer_data["total_orders"]
)

## Cell 6 — Dispute rate
Lower is better.

In [ ]:
buyer_data["dispute_rate"] = (
    buyer_data["disputes"]
    / buyer_data["total_orders"]
)

## Cell 7 — Payment reliability
Faster payment should produce a higher score.

In [ ]:
max_delay_days = max(
    buyer_data["average_payment_delay_days"].max(),
    1
)

buyer_data["payment_reliability"] = (
    1
    - (
        buyer_data["average_payment_delay_days"]
        / max_delay_days
    )
)

buyer_data["payment_reliability"] = (
    buyer_data["payment_reliability"]
    .clip(0, 1)
)

## Cell 8 — Reliability score
A transparent, explainable weighted score rather than an opaque model output.

In [ ]:
buyer_data["reliability_score"] = (
    0.40
    * buyer_data["completion_rate"]

    + 0.25
    * (1 - buyer_data["cancellation_rate"])

    + 0.20
    * buyer_data["payment_reliability"]

    + 0.15
    * (1 - buyer_data["dispute_rate"])
)

buyer_data["reliability_score_100"] = (
    buyer_data["reliability_score"]
    * 100
)

## Cell 9 — Risk classification

In [ ]:
def classify_risk(score):

    if score >= 80:
        return "Low Risk"

    elif score >= 60:
        return "Medium Risk"

    else:
        return "High Risk"

In [ ]:
buyer_data["risk_level"] = (
    buyer_data[
        "reliability_score_100"
    ].apply(classify_risk)
)

## Cell 10 — View buyers

In [ ]:
buyer_data[
    [
        "buyer_id",
        "market",
        "total_orders",
        "completion_rate",
        "cancellation_rate",
        "payment_reliability",
        "dispute_rate",
        "reliability_score_100",
        "risk_level"
    ]
].head(20)

## Cell 11 — Rank buyers

In [ ]:
buyer_ranking = (
    buyer_data
    .sort_values(
        "reliability_score_100",
        ascending=False
    )
    .reset_index(drop=True)
)

buyer_ranking["rank"] = (
    buyer_ranking.index + 1
)

## Cell 12 — Market-level buyer reliability
Important because Kadak Bhav recommends markets, not just individual buyers.

In [ ]:
market_reliability = (
    buyer_data
    .groupby("market")
    .agg(
        buyers=("buyer_id", "count"),
        avg_reliability=(
            "reliability_score_100",
            "mean"
        ),
        avg_completion_rate=(
            "completion_rate",
            "mean"
        ),
        avg_cancellation_rate=(
            "cancellation_rate",
            "mean"
        ),
        avg_dispute_rate=(
            "dispute_rate",
            "mean"
        )
    )
    .reset_index()
)

## Cell 13 — Market risk category

In [ ]:
market_reliability["risk_level"] = (
    market_reliability[
        "avg_reliability"
    ].apply(classify_risk)
)

market_reliability.sort_values(
    "avg_reliability",
    ascending=False
)

## Cell 14 — Reliability graph

In [ ]:
top = market_reliability.sort_values(
    "avg_reliability",
    ascending=False
)

plt.figure(figsize=(9, 5))

plt.bar(
    top["market"],
    top["avg_reliability"]
)

plt.xlabel("Market")
plt.ylabel("Average Reliability Score")
plt.title(
    "Buyer Reliability by Market"
)

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Cell 15 — Reliability distribution

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(
    buyer_data["reliability_score_100"],
    bins=10
)

plt.xlabel("Reliability Score")
plt.ylabel("Number of Buyers")
plt.title(
    "Buyer Reliability Score Distribution"
)

plt.tight_layout()
plt.show()

## Cell 16 — Risk distribution

In [ ]:
risk_counts = (
    buyer_data["risk_level"]
    .value_counts()
)

plt.figure(figsize=(7, 5))

plt.bar(
    risk_counts.index,
    risk_counts.values
)

plt.xlabel("Risk Level")
plt.ylabel("Number of Buyers")
plt.title(
    "Buyer Risk Distribution"
)

plt.tight_layout()
plt.show()

## Cell 17 — Connect Task 9 + Task 10
The most important part: bring the logistics-based recommendation (Task 9) and the buyer-reliability score (this notebook) together.

In [ ]:
logistics = pd.read_csv(
    "../data/processed/"
    "real_world_market_recommendations.csv"
)

In [ ]:
final_market_data = logistics.merge(
    market_reliability[
        [
            "market",
            "avg_reliability",
            "risk_level"
        ]
    ],
    on="market",
    how="left"
)

## Cell 18 — Replace old reliability penalty
Task 8/9 used a prototype reliability penalty. Now use the actual market-level reliability score computed above.

In [ ]:
risk_weight = 0.10

final_market_data[
    "reliability_penalty"
] = (
    (
        1
        - (
            final_market_data[
                "avg_reliability"
            ] / 100
        )
    )
    * risk_weight
    * final_market_data[
        "predicted_revenue"
    ]
)

## Cell 19 — Final net realization

In [ ]:
final_market_data[
    "final_net_realization"
] = (
    final_market_data[
        "predicted_revenue"
    ]

    - final_market_data[
        "transport_cost"
    ]

    - final_market_data[
        "perishability_penalty"
    ]

    - final_market_data[
        "reliability_penalty"
    ]
)

## Cell 20 — Rank final markets

In [ ]:
final_market_data = (
    final_market_data
    .sort_values(
        "final_net_realization",
        ascending=False
    )
    .reset_index(drop=True)
)

final_market_data["rank"] = (
    final_market_data.index + 1
)

## Cell 21 — Final recommendation

In [ ]:
best = final_market_data.iloc[0]

print("🏆 Recommended Market")
print("----------------------")
print("Market:",
      best["market"])

print(
    "Net Realization:",
    round(
        best["final_net_realization"],
        2
    )
)

print(
    "Buyer Reliability:",
    round(
        best["avg_reliability"],
        2
    )
)

print(
    "Risk:",
    best["risk_level"]
)

## Cell 22 — Save outputs

In [ ]:
os.makedirs(
    "../data/processed",
    exist_ok=True
)

buyer_data.to_csv(
    "../data/processed/"
    "buyer_reliability.csv",
    index=False
)

market_reliability.to_csv(
    "../data/processed/"
    "market_reliability.csv",
    index=False
)

final_market_data.to_csv(
    "../data/processed/"
    "final_market_recommendations.csv",
    index=False
)

## Optional ML layer — not yet
The transparent weighted score is deliberately preferred for the first version: it's explainable in an SIH Q&A ("we calculate buyer reliability from observable transaction behavior, not an opaque model prediction"). Once enough historical transactions accumulate, a model predicting probability of successful transaction (from orders, payment history, cancellations, disputes, delay, quantity, market, season) is a natural next step — but this task establishes the reliable scoring system first.